# Clip roads - flood scenarios

In [1]:
import arcpy
import os
import re

# =========================
# USER INPUTS
# =========================
map_name = "7.Flood+road"
group_layer_name = "Flood_scenarios"

clip_layer = r"E:\Paper\GIS\MyProject_paper\MyProject_paper.gdb\Pocomoke_closeroads6m__Project_noshare"

# Folder where outputs will be stored
output_folder = r"E:\01. Project_deliverable\GIS\Transportation\Road network\roads_inters"

# If you want shapefiles, keep this as False
# If you want feature classes inside a GDB, set True and provide gdb path below
use_gdb_output = False
output_gdb = r"E:\01. Project_deliverable\GIS\Transportation\Road network\roads_inters\roads_inters.gdb"

# =========================
# FUNCTIONS
# =========================
def clean_name(name, max_len=50):
    """
    Clean layer name for valid output name.
    """
    name = re.sub(r'[^A-Za-z0-9_]+', '_', name)
    name = re.sub(r'_+', '_', name).strip('_')
    return name[:max_len]

def get_group_layer(the_map, target_group_name):
    """
    Find the group layer by name.
    """
    for lyr in the_map.listLayers():
        if lyr.isGroupLayer and lyr.name == target_group_name:
            return lyr
    return None


# ENVIRONMENT
arcpy.env.overwriteOutput = True

# Create output folder if it does not exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# If using GDB output, create it if needed
if use_gdb_output:
    gdb_folder = os.path.dirname(output_gdb)
    gdb_name = os.path.basename(output_gdb)
    if not arcpy.Exists(output_gdb):
        arcpy.management.CreateFileGDB(gdb_folder, gdb_name)

# =========================
# ACCESS CURRENT PROJECT AND MAP
# =========================
aprx = arcpy.mp.ArcGISProject("CURRENT")
m = None

for mp in aprx.listMaps():
    if mp.name == map_name:
        m = mp
        break

if m is None:
    raise ValueError(f"Map '{map_name}' was not found in the current ArcGIS Pro project.")

group_layer = get_group_layer(m, group_layer_name)

if group_layer is None:
    raise ValueError(f"Group layer '{group_layer_name}' was not found in map '{map_name}'.")

# =========================
# PROCESS EACH FLOOD SCENARIO
# =========================
layers_in_group = group_layer.listLayers()

if not layers_in_group:
    raise ValueError(f"No layers found inside group layer '{group_layer_name}'.")

print(f"Found {len(layers_in_group)} layer(s) inside '{group_layer_name}'.\n")

for flood_lyr in layers_in_group:
    # Skip broken or non-feature layers if needed
    if flood_lyr.isGroupLayer:
        print(f"Skipping subgroup layer: {flood_lyr.name}")
        continue

    try:
        desc = arcpy.Describe(flood_lyr)
        if not hasattr(desc, "shapeType"):
            print(f"Skipping non-feature layer: {flood_lyr.name}")
            continue
    except Exception:
        print(f"Skipping unreadable layer: {flood_lyr.name}")
        continue

    flood_name_clean = clean_name(flood_lyr.name)
    out_name = f"{flood_name_clean}_road"

    if use_gdb_output:
        out_path = os.path.join(output_gdb, out_name)
    else:
        out_path = os.path.join(output_folder, f"{out_name}.shp")

    print(f"Clipping roads with: {flood_lyr.name}")
    print(f"Output: {out_path}")

    arcpy.analysis.Clip(
        in_features=clip_layer,
        clip_features=flood_lyr,
        out_feature_class=out_path
    )

    print("Done.\n")

print("All clips finished successfully.")

Found 6 layer(s) inside 'Flood_scenarios'.

Clipping roads with: 2yearflood
Output: E:\01. Project_deliverable\GIS\Transportation\Road network\roads_inters\2yearflood_road.shp
Done.

Clipping roads with: 10yearflood
Output: E:\01. Project_deliverable\GIS\Transportation\Road network\roads_inters\10yearflood_road.shp
Done.

Clipping roads with: 100yearflood
Output: E:\01. Project_deliverable\GIS\Transportation\Road network\roads_inters\100yearflood_road.shp
Done.

Clipping roads with: GDB_future_2yearflood
Output: E:\01. Project_deliverable\GIS\Transportation\Road network\roads_inters\GDB_future_2yearflood_road.shp
Done.

Clipping roads with: GDB_future_10yearflood
Output: E:\01. Project_deliverable\GIS\Transportation\Road network\roads_inters\GDB_future_10yearflood_road.shp
Done.

Clipping roads with: GDB_future_100yearflood
Output: E:\01. Project_deliverable\GIS\Transportation\Road network\roads_inters\GDB_future_100yearflood_road.shp
Done.

All clips finished successfully.


# Compare ROADS future and current

In [ ]:
import arcpy
import os
import re

# ============================================
# USER INPUTS
# ============================================
input_folder = r"E:\01. Project_deliverable\GIS\Transportation\Road network\roads_inters"
output_folder = r"E:\01. Project_deliverable\GIS\Transportation\Road network\roads_inters\comparison"

# ============================================
# ENVIRONMENT
# ============================================
arcpy.env.overwriteOutput = True

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# ============================================
# HELPER FUNCTIONS
# ============================================
def list_shapefiles(folder):
    """List all shapefiles in the input folder."""
    return [
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith(".shp")
    ]

def extract_return_period(name):
    """
    Extract return period such as:
    2yr, 10yr, 100yr
    """
    n = name.lower()

    patterns = [
        r'(\d+)\s*yr',
        r'(\d+)\s*year'
    ]

    for p in patterns:
        m = re.search(p, n)
        if m:
            return f"{m.group(1)}yr"

    return None

def extract_time_scenario(name):
    """
    Detect current or future from file name.
    """
    n = name.lower()

    if "current" in n:
        return "current"
    elif "future" in n:
        return "future"
    else:
        return None

def clean_name(name, max_len=50):
    """
    Clean text for shapefile naming.
    """
    name = re.sub(r'[^A-Za-z0-9_]+', '_', name)
    name = re.sub(r'_+', '_', name).strip('_')
    return name[:max_len]

# ============================================
# READ INPUT FILES
# ============================================
feature_files = list_shapefiles(input_folder)

if not feature_files:
    raise ValueError(f"No shapefiles found in: {input_folder}")

print(f"Found {len(feature_files)} shapefile(s).")

# Organize files by return period
paired = {}

for fc in feature_files:
    base = os.path.basename(fc)
    name_no_ext = os.path.splitext(base)[0]

    rp = extract_return_period(name_no_ext)
    scen = extract_time_scenario(name_no_ext)

    if rp is None or scen is None:
        print(f"Skipping file: {base} (could not identify return period or scenario)")
        continue

    if rp not in paired:
        paired[rp] = {}

    paired[rp][scen] = fc

print("\nDetected pairs:")
for rp, vals in paired.items():
    print(rp, vals)

In [ ]:
# ============================================
# CREATE FUTURE - CURRENT FILES
# ============================================
created_files = []

for rp in sorted(paired.keys(), key=lambda x: int(re.findall(r'\d+', x)[0])):
    current_fc = paired[rp].get("current")
    future_fc = paired[rp].get("future")

    if not current_fc or not future_fc:
        print(f"\nSkipping {rp}: missing current or future file.")
        continue

    out_name = clean_name(f"{rp}_future_minus_current") + ".shp"
    out_path = os.path.join(output_folder, out_name)

    print(f"\nProcessing return period: {rp}")
    print(f"Future layer : {os.path.basename(future_fc)}")
    print(f"Current layer: {os.path.basename(current_fc)}")
    print(f"Output       : {out_path}")

    # Erase current from future
    arcpy.analysis.Erase(
        in_features=future_fc,
        erase_features=current_fc,
        out_feature_class=out_path
    )

    created_files.append(out_path)
    print("Done.")

print("\n===================================")
print("Finished creating future-current layers.")
print("Created files:")

for f in created_files:
    print(f)

# compare roads flooded (model) x community experiences

In [5]:
import arcpy
import os
import csv

# =========================================================
# USER INPUTS
# =========================================================
vulnerable_roads = r"E:\Paper\GIS\MyProject_paper\MyProject_paper.gdb\Vulnerableroads_Buff_Project"

flood_layers = {
    "2yr_current": r"E:\01. Project_deliverable\GIS\Transportation\Road network\roads_inters\2yearflood_road_current.shp",
    "10yr_current": r"E:\01. Project_deliverable\GIS\Transportation\Road network\roads_inters\10yearflood_road_current.shp",
    "100yr_current": r"E:\01. Project_deliverable\GIS\Transportation\Road network\roads_inters\100yearflood_road_current.shp"
}

output_folder = r"E:\01. Project_deliverable\GIS\Transportation\Road network\vulnerable_roads_intersection\comparison"

# Set True if you also want Excel output
create_excel = True

# =========================================================
# ENVIRONMENT
# =========================================================
arcpy.env.overwriteOutput = True

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# =========================================================
# HELPER FUNCTIONS
# =========================================================
def get_total_area(fc):
    total = 0.0
    with arcpy.da.SearchCursor(fc, ["SHAPE@AREA"]) as cursor:
        for row in cursor:
            if row[0] is not None:
                total += row[0]
    return total

def get_count(fc):
    return int(arcpy.management.GetCount(fc)[0])

def get_area_unit_label(fc):
    desc = arcpy.Describe(fc)
    sr = desc.spatialReference
    try:
        unit_name = sr.linearUnitName.lower()
        if "meter" in unit_name:
            return "square meters"
        elif "foot" in unit_name or "feet" in unit_name:
            return "square feet"
        else:
            return f"square {sr.linearUnitName}"
    except:
        return "map units squared"

# =========================================================
# PREPARE BASE INFO
# =========================================================
if not arcpy.Exists(vulnerable_roads):
    raise ValueError(f"Vulnerable roads layer not found:\n{vulnerable_roads}")

total_vuln_area = get_total_area(vulnerable_roads)
total_vuln_count = get_count(vulnerable_roads)
area_unit = get_area_unit_label(vulnerable_roads)

print("Base vulnerable roads layer:")
print(vulnerable_roads)
print(f"Total polygon count: {total_vuln_count}")
print(f"Total area: {total_vuln_area:.2f} {area_unit}\n")

results = []

# =========================================================
# PROCESS EACH FLOOD LAYER
# =========================================================
for scenario_name, flood_fc in flood_layers.items():
    print(f"Processing: {scenario_name}")

    if not arcpy.Exists(flood_fc):
        print(f"  Flood layer not found, skipping:\n  {flood_fc}\n")
        continue

    # Create temporary feature layer from vulnerable roads
    vuln_lyr = f"vuln_lyr_{scenario_name}"
    arcpy.management.MakeFeatureLayer(vulnerable_roads, vuln_lyr)

    # Select vulnerable roads that intersect the flood layer
    arcpy.management.SelectLayerByLocation(
        in_layer=vuln_lyr,
        overlap_type="INTERSECT",
        select_features=flood_fc,
        selection_type="NEW_SELECTION"
    )

    selected_count = get_count(vuln_lyr)
    selected_area = get_total_area(vuln_lyr)

    if total_vuln_area > 0:
        percent_affected = (selected_area / total_vuln_area) * 100
    else:
        percent_affected = None

    # Export selected features to shapefile
    out_name = f"VulnerableRoads_{scenario_name}_intersect.shp"
    out_path = os.path.join(output_folder, out_name)

    arcpy.management.CopyFeatures(vuln_lyr, out_path)

    results.append({
        "scenario": scenario_name,
        "flood_layer": os.path.basename(flood_fc),
        "total_vulnerable_polygon_count": total_vuln_count,
        "selected_vulnerable_polygon_count": selected_count,
        "total_vulnerable_area": total_vuln_area,
        "selected_vulnerable_area": selected_area,
        "percent_vulnerable_area_affected": percent_affected,
        "output_shapefile": out_path
    })

    print(f"  Selected polygon count: {selected_count}")
    print(f"  Selected area: {selected_area:.2f} {area_unit}")
    if percent_affected is not None:
        print(f"  Percent affected: {percent_affected:.2f}%")
    else:
        print("  Percent affected: undefined")
    print(f"  Output shapefile: {out_path}\n")

    # Clean selection and temp layer
    arcpy.management.Delete(vuln_lyr)

# =========================================================
# SAVE CSV SUMMARY
# =========================================================
csv_path = os.path.join(output_folder, "vulnerable_roads_flood_intersection_summary.csv")

fieldnames = [
    "scenario",
    "flood_layer",
    "total_vulnerable_polygon_count",
    "selected_vulnerable_polygon_count",
    "total_vulnerable_area",
    "selected_vulnerable_area",
    "percent_vulnerable_area_affected",
    "output_shapefile"
]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

print(f"CSV summary saved to:\n{csv_path}\n")

# =========================================================
# OPTIONAL EXCEL OUTPUT
# =========================================================
if create_excel:
    try:
        from openpyxl import Workbook

        xlsx_path = os.path.join(output_folder, "vulnerable_roads_flood_intersection_summary.xlsx")

        wb = Workbook()
        ws = wb.active
        ws.title = "summary"

        ws.append(fieldnames)
        for row in results:
            ws.append([row[f] for f in fieldnames])

        wb.save(xlsx_path)
        print(f"Excel summary saved to:\n{xlsx_path}\n")

    except Exception as e:
        print(f"Excel file not created. Reason: {e}\n")

# =========================================================
# PRINT FINAL INTERPRETATION
# =========================================================
print("===== FINAL SUMMARY =====")
for row in results:
    if row["percent_vulnerable_area_affected"] is not None:
        print(
            f"{row['scenario']}: "
            f"{row['percent_vulnerable_area_affected']:.2f}% of vulnerable-road buffer area "
            f"is represented by polygons that intersect the flooded-road layer."
        )
    else:
        print(
            f"{row['scenario']}: percentage could not be calculated because total vulnerable area is zero."
        )

print("\nFinished.")

Base vulnerable roads layer:
E:\Paper\GIS\MyProject_paper\MyProject_paper.gdb\Vulnerableroads_Buff_Project
Total polygon count: 40
Total area: 1501210.41 square feet

Processing: 2yr_current
  Selected polygon count: 18
  Selected area: 869178.07 square feet
  Percent affected: 57.90%
  Output shapefile: E:\01. Project_deliverable\GIS\Transportation\Road network\vulnerable_roads_intersection\comparison\VulnerableRoads_2yr_current_intersect.shp

Processing: 10yr_current
  Selected polygon count: 26
  Selected area: 1116644.52 square feet
  Percent affected: 74.38%
  Output shapefile: E:\01. Project_deliverable\GIS\Transportation\Road network\vulnerable_roads_intersection\comparison\VulnerableRoads_10yr_current_intersect.shp

Processing: 100yr_current
  Selected polygon count: 33
  Selected area: 1364936.61 square feet
  Percent affected: 90.92%
  Output shapefile: E:\01. Project_deliverable\GIS\Transportation\Road network\vulnerable_roads_intersection\comparison\VulnerableRoads_100yr_cu

# comparison roads flooded (model and community x city total road)

In [10]:
import arcpy

arcpy.env.overwriteOutput = True


# INPUTS
city_road = r"E:\Paper\GIS\MyProject_paper\MyProject_paper.gdb\Pocomoke_closeroads6m__Project_noshare"
vulnerable_roads = r"E:\Paper\GIS\MyProject_paper\MyProject_paper.gdb\Vulnerableroads_Buff_Project"
flood_layers = {
    "2yr_current": r"E:\01. Project_deliverable\GIS\Transportation\Road network\roads_inters\2yearflood_road_current.shp",
    "2yr_future": r"E:\01. Project_deliverable\GIS\Transportation\Road network\roads_inters\2yearflood_road_future.shp",
    "10yr_current": r"E:\01. Project_deliverable\GIS\Transportation\Road network\roads_inters\10yearflood_road_current.shp",
    "10yr_future": r"E:\01. Project_deliverable\GIS\Transportation\Road network\roads_inters\10yearflood_road_future.shp",
    "100yr_current": r"E:\01. Project_deliverable\GIS\Transportation\Road network\roads_inters\100yearflood_road_current.shp",
    "100yr_future": r"E:\01. Project_deliverable\GIS\Transportation\Road network\roads_inters\100yearflood_road_future.shp"
}


# FUNCTIONS
def total_area(fc):
    area = 0.0
    with arcpy.da.SearchCursor(fc, ["SHAPE@AREA"]) as cursor:
        for row in cursor:
            if row[0] is not None:
                area += row[0]
    return area

def area_unit_label(fc):
    sr = arcpy.Describe(fc).spatialReference
    try:
        unit_name = sr.linearUnitName.lower()
        if "foot" in unit_name or "feet" in unit_name:
            return "square feet"
        elif "meter" in unit_name:
            return "square meters"
        else:
            return f"square {sr.linearUnitName}"
    except:
        return "map units squared"

In [11]:
# 1) TOTAL AREA OF CITY ROAD BUFFERS

city_road_area = total_area(city_road)
units = area_unit_label(city_road)

if city_road_area == 0:
    raise ValueError("Total city road buffer area is zero. Check the input layer.")

print("===== 1) TOTAL CITY ROAD BUFFER AREA =====")
print(f"Total city road buffer area = {city_road_area:.2f} {units}\n")

# 2) TOTAL AREA OF FLOODED ROAD BUFFERS

print("===== 2) FLOODED ROAD BUFFER AREA BY RETURN PERIOD =====")

flood_results = {}

for scenario, flood_fc in flood_layers.items():
    flood_area = total_area(flood_fc)
    flood_results[scenario] = flood_area
    print(f"{scenario} = {flood_area:.2f} {units}")

print()

# 3) TOTAL AREA OF VULNERABLE ROAD BUFFERS
vulnerable_area = total_area(vulnerable_roads)

print("===== 3) TOTAL VULNERABLE ROAD BUFFER AREA =====")
print(f"Vulnerable road buffer area = {vulnerable_area:.2f} {units}\n")


# 4) PERCENTAGES RELATIVE TO TOTAL CITY ROAD BUFFER
print("===== 4) PERCENTAGE RELATIVE TO TOTAL CITY ROAD BUFFER =====")

for scenario, flood_area in flood_results.items():
    flood_pct = (flood_area / city_road_area) * 100
    print(f"{scenario} / total city road area = {flood_pct:.2f}%")

vulnerable_pct = (vulnerable_area / city_road_area) * 100
print(f"Vulnerable roads / total city road area = {vulnerable_pct:.2f}%")

===== 1) TOTAL CITY ROAD BUFFER AREA =====
Total city road buffer area = 8948524.55 square feet

===== 2) FLOODED ROAD BUFFER AREA BY RETURN PERIOD =====
2yr_current = 294552.38 square feet
2yr_future = 342401.72 square feet
10yr_current = 698417.56 square feet
10yr_future = 825172.01 square feet
100yr_current = 1580933.44 square feet
100yr_future = 1756908.72 square feet

===== 3) TOTAL VULNERABLE ROAD BUFFER AREA =====
Vulnerable road buffer area = 1501210.41 square feet

===== 4) PERCENTAGE RELATIVE TO TOTAL CITY ROAD BUFFER =====
2yr_current / total city road area = 3.29%
2yr_future / total city road area = 3.83%
10yr_current / total city road area = 7.80%
10yr_future / total city road area = 9.22%
100yr_current / total city road area = 17.67%
100yr_future / total city road area = 19.63%
Vulnerable roads / total city road area = 16.78%
